<a href="https://colab.research.google.com/github/promckkon/MK2DimCNN/blob/main/MK-DCNN%20with%200dB%20NOISE%20in%20CWRU%20Dataset%20260504.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================================================
# Cell 1: 環境配置與套件載入 (Environment & Setup)
# =========================================================
import os
import warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore') # 隱藏 Pandas FutureWarning

# 自動檢查並修復 CatBoost 與 NumPy 版本衝突
try:
    import catboost
    import numpy as np
    if catboost.__version__ != '1.2.7' or np.__version__ >= '2.0.0':
        raise ImportError
except:
    print("正在安裝穩定版環境 (NumPy 1.26.4 & CatBoost 1.2.7)...")
    get_ipython().system('pip install -q "numpy==1.26.4" "catboost==1.2.7" "scipy" "scikit-learn" "seaborn" "matplotlib"')
    print("--- 安裝完成！請點擊上方選單「執行階段」->「重新啟動執行階段」，然後再次執行此單元格 ---")

import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.manifold import TSNE
from sklearn.utils import resample
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis
from datetime import datetime
import random

print("所有套件載入完成，準備就緒！")

In [ ]:
# =========================================================
# Cell 2: 資料讀取、動態切割、特徵提取與平衡
# =========================================================
# 1. 讀取數據
DATASET_PATH = '/content/drive/MyDrive/MK-DCNN CWRU/NOISE_0_faults.csv'
if not os.path.exists(DATASET_PATH):
    print(f"錯誤：找不到檔案 {DATASET_PATH}")
else:
    df = pd.read_csv(DATASET_PATH)

    # 2. 動態計算滑動視窗參數
    TARGET_ROWS = 1800
    total_samples = len(df)
    fault_types = df['fault'].unique()
    num_faults = len(fault_types)

    avg_samples_per_fault = total_samples / num_faults
    approx_windows = TARGET_ROWS / num_faults
    win_len = int(avg_samples_per_fault / approx_windows)
    stride = int(win_len * 0.8)
    print(f"動態計算完成 -> WINDOW_SIZE: {win_len}, STRIDE: {stride}")

    # 3. 同步提取 Raw Data (給 CNN) 與 統計特徵，確保特徵對齊
    X_raw_list, stats_list, y_list = [], [], []

    for f in fault_types:
        fault_data = df[df['fault'] == f]['DE_data'].values.astype(float)
        num_windows = (len(fault_data) - win_len) // stride + 1

        for i in range(num_windows):
            window = fault_data[i*stride : i*stride + win_len]

            # 保存給 CNN 的 Raw Data
            X_raw_list.append(window.reshape(-1, 1))
            y_list.append(f)

            # 計算統計特徵
            rms_val = np.sqrt(np.mean(np.square(window)))
            mean_abs = np.mean(np.abs(window))
            stats_list.append([
                np.mean(window), np.std(window), rms_val, np.max(window), np.min(window),
                skew(window), kurtosis(window),
                rms_val / mean_abs if mean_abs != 0 else 0,
                np.max(window) / rms_val if rms_val != 0 else 0
            ])

    # 4. 資料平衡處理 (Resample to exact 1570)
    temp_df = pd.DataFrame({'X_raw': X_raw_list, 'stats': stats_list, 'fault': y_list})
    TARGET_BALANCED_ROWS = 1570
    samples_per_class = TARGET_BALANCED_ROWS // num_faults

    resampled_data = []
    for f in fault_types:
        class_data = temp_df[temp_df['fault'] == f]
        replace_flag = len(class_data) < samples_per_class
        resampled = resample(class_data, replace=replace_flag, n_samples=samples_per_class, random_state=42)
        resampled_data.append(resampled)

    balanced_df = pd.concat(resampled_data).sample(frac=1, random_state=42).reset_index(drop=True)

    # 拆解回 NumPy Array
    X_raw = np.stack(balanced_df['X_raw'].values)
    X_stat = np.stack(balanced_df['stats'].values)
    y_label = balanced_df['fault'].values

    # 標籤編碼
    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y_label)
    y_categorical = to_categorical(y_encoded)

    print(f"資料平衡與對齊完成！總筆數: {len(balanced_df)}")

In [ ]:
# =========================================================
# Cell 3: 物理約束損失函數與 MK-DCNN 模型訓練
# =========================================================
def custom_loss(y_true, y_pred):
    loss = K.categorical_crossentropy(y_true, y_pred)
    if K.int_shape(y_pred)[-1] > 1:
        diff = y_pred[:, 1:] - y_pred[:, :-1]
        physics_term = tf.reduce_mean(tf.square(diff))
    else:
        physics_term = 0.0
    return loss + 0.01 * physics_term

input_shape = (X_raw.shape[1], 1)
no_classes = len(encoder.classes_)

# 建立 MK-DCNN 架構
inputs = layers.Input(shape=input_shape)
# Head 1
conv1 = layers.Conv1D(64, 200, activation='relu')(inputs)
drop1 = layers.Dropout(0.5)(conv1)
flat1 = layers.Flatten()(layers.MaxPooling1D(20)(drop1))
# Head 2
conv2 = layers.Conv1D(64, 100, activation='relu')(inputs)
drop2 = layers.Dropout(0.5)(conv2)
flat2 = layers.Flatten()(layers.MaxPooling1D(10)(drop2))
# Head 3
conv3 = layers.Conv1D(64, 50, activation='relu')(inputs)
drop3 = layers.Dropout(0.5)(conv3)
flat3 = layers.Flatten()(layers.MaxPooling1D(5)(drop3))

merged = layers.concatenate([flat1, flat2, flat3])
dense1 = layers.Dense(100, activation='relu', name='deep_feature_layer')(merged)
outputs = layers.Dense(no_classes, activation='softmax')(dense1)

cnn_model = models.Model(inputs=inputs, outputs=outputs)
cnn_model.compile(optimizer='adam', loss=custom_loss, metrics=['accuracy'])

# 進行初步訓練以獲取有意義的特徵
print("開始訓練 MK-DCNN...")
X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(X_raw, y_categorical, test_size=0.3, random_state=42)
cnn_model.fit(X_train_cnn, y_train_cnn, batch_size=100, epochs=20, verbose=1, validation_data=(X_test_cnn, y_test_cnn))

# 擷取深度特徵
print("提取 Dense(100) 深度特徵...")
dummy_cnn = models.Model(inputs=cnn_model.input, outputs=cnn_model.get_layer('deep_feature_layer').output)
X_deep = dummy_cnn.predict(X_raw)

In [ ]:
# =========================================================
# Cell 4: 特徵正規化、t-SNE 降維與雙路徑特徵融合 (Feature Fusion)
# =========================================================
# 1. 統計特徵標準化與降維
scaler = StandardScaler()
X_stat_scaled = scaler.fit_transform(X_stat)

print("正在對「統計特徵」執行 t-SNE (降至 2 維)...")
tsne_stat = TSNE(n_components=2, perplexity=40, random_state=42)
X_stat_2d = tsne_stat.fit_transform(X_stat_scaled)

# 2. 深度特徵降維
print("正在對「深度特徵」執行 t-SNE (降至 2 維)...")
tsne_deep = TSNE(n_components=2, perplexity=40, random_state=42)
X_deep_2d = tsne_deep.fit_transform(X_deep)

# 3. 雙路徑特徵融合 (2D + 2D = 4D)
X_fused = np.concatenate((X_stat_2d, X_deep_2d), axis=1)

print(f"特徵融合完成！最終 CatBoost 輸入維度: {X_fused.shape} (包含統計 t-SNE 與 深度 t-SNE)")

# 切分最終用於 HS-PSO 與 CatBoost 的數據集
X_train, X_test, y_train, y_test = train_test_split(X_fused, y_encoded, test_size=0.2, random_state=42)

In [ ]:
# =========================================================
# Cell 5: 封裝 HS-PSO 最佳化演算法 (物件導向架構)
# =========================================================
from catboost import CatBoostClassifier

class HSPSO_Optimizer:
    def __init__(self, X_tr, y_tr, X_va, y_va, search_space, n_particles=10, seed=42):
        self.X_tr, self.y_tr = X_tr, y_tr
        self.X_va, self.y_va = X_va, y_va
        self.space = search_space
        self.keys = list(search_space.keys())
        self.n_particles = n_particles
        self.rng = np.random.default_rng(seed)

        # HS-PSO 參數
        self.w_min, self.w_max = 0.5, 0.9
        self.c1, self.c2 = 0.8, 0.6
        self.mutation_prob = 0.15

        self.swarm = [self._random_particle() for _ in range(n_particles)]
        self.pbest = self.swarm.copy()
        self.pbest_scores = [self._evaluate(p) for p in self.swarm]

        best_idx = int(np.argmax(self.pbest_scores))
        self.gbest = self.pbest[best_idx].copy()
        self.gbest_score = self.pbest_scores[best_idx]

    def _random_particle(self):
        return {k: self.rng.choice(self.space[k]) for k in self.keys}

    def _project_to_space(self, key, value):
        choices = np.array(self.space[key], dtype=float)
        idx = int(np.argmin(np.abs(choices - float(value))))
        v = self.space[key][idx]
        return int(v) if isinstance(self.space[key][0], int) else float(v)

    def _evaluate(self, params):
        model = CatBoostClassifier(
            iterations=int(params["iterations"]), depth=int(params["depth"]),
            learning_rate=float(params["learning_rate"]), l2_leaf_reg=float(params["l2_leaf_reg"]),
            bagging_temperature=float(params["bagging_temperature"]), random_strength=float(params["random_strength"]),
            loss_function="MultiClass", eval_metric="Accuracy",
            od_type="Iter", od_wait=20, verbose=False, random_seed=42, thread_count=-1
        )
        model.fit(self.X_tr, self.y_tr, eval_set=(self.X_va, self.y_va), use_best_model=True)
        pred = model.predict(self.X_va).reshape(-1)
        return float(accuracy_score(self.y_va, pred))

    def _cauchy_mutation(self, particle, gamma=0.25):
        mutated = dict(particle)
        for k in self.keys:
            factor = 1.0 + gamma * np.tan(np.pi * (self.rng.random() - 0.5))
            new_val = float(mutated[k]) * factor
            mutated[k] = self._project_to_space(k, new_val)
        return mutated

    def optimize(self, max_iter=15, target_acc=0.999):
        print(f"🚀 開始 HS-PSO 尋優 (粒子數={self.n_particles}, 迭代={max_iter})...")
        for it in range(1, max_iter + 1):
            if self.gbest_score >= target_acc:
                print("達到目標準確率，提前停止！")
                break

            w = self.w_max - (self.w_max - self.w_min) * (it / max_iter)**2 # 非線性慣性

            for i in range(self.n_particles):
                # 簡化的離散 PSO 速度/位置更新概念 (以機率跟隨)
                for k in self.keys:
                    r1, r2 = self.rng.random(), self.rng.random()
                    if r1 < 0.4:
                        self.swarm[i][k] = self.pbest[i][k]
                    elif r2 < 0.4:
                        self.swarm[i][k] = self.gbest[k]

                # 柯西變異
                if self.rng.random() < self.mutation_prob:
                    self.swarm[i] = self._cauchy_mutation(self.swarm[i])

                score = self._evaluate(self.swarm[i])
                if score > self.pbest_scores[i]:
                    self.pbest[i] = self.swarm[i].copy()
                    self.pbest_scores[i] = score
                    if score > self.gbest_score:
                        self.gbest = self.swarm[i].copy()
                        self.gbest_score = score

            print(f"迭代 {it}/{max_iter} - 最佳驗證準確率: {self.gbest_score:.4f}")

        print(f"\n✅ 尋優完成！最佳參數: {self.gbest}")
        return self.gbest, self.gbest_score

In [ ]:
# =========================================================
# Cell 6: 模型部署、評估報告與視覺化圖表
# =========================================================
from catboost import Pool

search_space = {
    "iterations": list(range(50, 201, 25)),
    "depth": list(range(2, 9)),
    "learning_rate": [0.03, 0.05, 0.08, 0.1],
    "l2_leaf_reg": [1.0, 3.0, 5.0],
    "bagging_temperature": [0.5, 1.0, 1.5],
    "random_strength": [0.5, 1.0, 1.5]
}

# 1. 將訓練資料再次切分以供 PSO 驗證
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, va_idx = next(sss.split(X_train, y_train))
X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
X_va, y_va = X_train[va_idx], y_train[va_idx]

# 2. 執行 HS-PSO 最佳化
start_time = datetime.now()
hs_pso = HSPSO_Optimizer(X_tr, y_tr, X_va, y_va, search_space, n_particles=10)
best_params, best_val_acc = hs_pso.optimize(max_iter=15)
print('Tuning Duration:', datetime.now() - start_time)

# 3. 部署最終 CatBoost 模型
train_pool = Pool(X_train, y_train)
test_pool = Pool(X_test, y_test)

final_model = CatBoostClassifier(
    iterations=max(300, int(best_params["iterations"]) * 2),
    depth=int(best_params["depth"]),
    learning_rate=float(best_params["learning_rate"]),
    l2_leaf_reg=float(best_params["l2_leaf_reg"]),
    bagging_temperature=float(best_params["bagging_temperature"]),
    random_strength=float(best_params["random_strength"]),
    loss_function="MultiClass", eval_metric="Accuracy",
    od_type="Iter", od_wait=40, verbose=50, random_seed=42, thread_count=-1
)

final_model.fit(train_pool, eval_set=test_pool, use_best_model=True)

# 4. 繪製 Accuracy Learning Curve
curve = final_model.get_evals_result()
train_acc = curve.get("learn", {}).get("Accuracy", [])
val_acc = curve.get("validation", {}).get("Accuracy", [])

if train_acc and val_acc:
    plt.figure(figsize=(8, 5))
    plt.plot(train_acc, label="Train Accuracy")
    plt.plot(val_acc, label="Val Accuracy")
    plt.title("Accuracy Learning Curve")
    plt.xlabel("Iterations")
    plt.ylabel("Accuracy")
    plt.grid(True)
    plt.legend()
    plt.show()


In [ ]:
# 5. 輸出 Classification Report 與 Confusion Matrix
y_pred_train = final_model.predict(X_train).reshape(-1)
y_pred_test = final_model.predict(X_test).reshape(-1)

print("\nClassification Report - Test Set:")
print(classification_report(y_test, y_pred_test, target_names=encoder.classes_))

cm_train = confusion_matrix(y_train, y_pred_train)
cm_test = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
sns.heatmap(cm_train, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title("Confusion Matrix - Training Set")
plt.xlabel("Predicted")
plt.ylabel("True")

plt.subplot(1, 2, 2)
sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title("Confusion Matrix - Test Set")
plt.xlabel("Predicted")
plt.ylabel("True")

plt.tight_layout()
plt.show()